# 13 · Introducción al Web Scraping

## Observatorio de auditoría · Diagnóstico de un portal de avisos

**Objetivo.** Localizar el dato antes de elegir una técnica, documentar su origen y distinguir una extracción fiable de una incompleta.

**Cómo trabajar.** Abre el notebook desde la carpeta extraída y ejecuta las celdas en orden con Python 3.10 o superior. Los ejemplos usan biblioteca estándar y un servidor temporal en 127.0.0.1. Todos los avisos son ficticios. No se consulta Internet.

**Material de repaso.** El ejemplo 6 documenta la demostración que el docente muestra en pantalla. Los diez ejercicios quedan para consulta o consolidación posterior. En los próximos bloques se implementarán la extracción de HTML y la automatización con Selenium.

**Regla de trabajo.** Una observación técnica identifica dónde aparece el dato. Cualquier uso sobre una fuente externa requiere comprobar el alcance autorizado, las reglas de acceso, el volumen y la minimización de datos.

**Versión alumnado.** Los ejemplos están resueltos y los ejercicios contienen espacios para tu código.

In [1]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import parse_qs, urlencode, urljoin, urlsplit
from urllib.request import ProxyHandler, build_opener
import json
import sys

BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
from servidor_aula import portal


def observar(base, ruta):
    # El portal de aula se consulta directamente, sin usar el proxy del entorno.
    cliente = build_opener(ProxyHandler({}))
    url = urljoin(base, ruta)
    try:
        respuesta = cliente.open(url, timeout=4)
    except HTTPError as error:
        respuesta = error
    with respuesta:
        cuerpo = respuesta.read()
        return {
            "url": respuesta.geturl(),
            "estado": respuesta.code,
            "tipo": respuesta.headers.get_content_type(),
            "charset": respuesta.headers.get_content_charset(),
            "bytes": cuerpo,
            "captura_utc": datetime.now(timezone.utc).isoformat(),
        }


def texto(respuesta):
    return respuesta["bytes"].decode(respuesta["charset"] or "utf-8")


## Ejemplo 1 · Formular una pregunta de observación

Antes de programar, define un universo y una unidad. Para el Observatorio, la unidad es un aviso con identificador estable. Una ejecución solo cubre las páginas y filtros acordados. Los títulos ayudan a leer, pero pueden cambiar sin que cambie el ID.

**Resultado esperado.** Lista corta de campos y una ventana explícita. El instante de captura se guardará aparte.

In [2]:
alcance = {
    "unidad": "aviso publicado",
    "periodo": "2026-10",
    "campos": ["id", "fecha", "titulo"],
    "fuente": "portal ficticio",
}
print(alcance)

{'unidad': 'aviso publicado', 'periodo': '2026-10', 'campos': ['id', 'fecha', 'titulo'], 'fuente': 'portal ficticio'}


## Ejemplo 2 · Separar ruta, consulta y fragmento

La query puede transportar filtros y la ruta identifica un recurso. El fragmento puede orientar la navegación o ser leído por JavaScript, pero no se envía como parte de la petición HTTP. El ejercicio trabaja con una URL ilustrativa y evita contactarla.

**Resultado esperado.** Una ruta, un diccionario de filtros y un fragmento separado.

In [3]:
consulta = urlencode({"anio": "2026", "q": "control & revisión"})
url = f"https://portal.example/avisos?{consulta}#resultados"
partes = urlsplit(url)
print(partes.path, parse_qs(partes.query), partes.fragment)

/avisos {'anio': ['2026'], 'q': ['control & revisión']} resultados


## Ejemplo 3 · Observar cabeceras y cuerpo

Un cliente HTTP recibe bytes. La cabecera Content-Type indica el formato y, en este portal, el parámetro charset especifica UTF-8. Este diagnóstico no ejecuta JavaScript ni usa un parser de extracción. Buscar un ID conocido es una sonda puntual.

**Resultado esperado.** 200, text/html y presencia de AV-101.

In [4]:
with portal() as base:
    respuesta = observar(base, "/estatico")
print(respuesta["estado"], respuesta["tipo"], respuesta["charset"])
print("AV-101" in texto(respuesta))

200 text/html utf-8
True


## Ejemplo 4 · Comparar HTML inicial en dos rutas

El mismo aviso puede aparecer en una ruta estática y en otra después de una interacción. La cabecera HTML de la tabla es compatible con ambos diseños. La ausencia del ID en el HTML dinámico confirma que el cuerpo observado no lo trae.

**Resultado esperado.** AV-101 está en el HTML estático, falta en el dinámico y este incluye un contenedor vacío.

In [5]:
with portal() as base:
    estatico = texto(observar(base, "/estatico"))
    dinamico = texto(observar(base, "/dinamico"))
print("AV-101" in estatico, "AV-101" in dinamico)
print('<tbody id="filas"></tbody>' in dinamico)

True False
True


## Ejemplo 5 · Reconocer un recurso posterior

El portal ficticio pide JSON cuando se pulsa el botón. En este laboratorio se permite consultar /datos.json directamente para inspeccionar el formato. En fuentes externas, una petición descubierta en DevTools puede carecer de contrato o permiso de reutilización.

**Resultado esperado.** application/json con AV-101 y AV-102. Estos registros se convierten en filas en el navegador tras la acción.

In [6]:
with portal() as base:
    recurso = observar(base, "/datos.json")
assert recurso["tipo"] == "application/json"
avisos = json.loads(recurso["bytes"])["items"]
print([aviso["id"] for aviso in avisos])

['AV-101', 'AV-102']


## Ejemplo 6 · Guion de DevTools para la demostración docente

1. Desde la carpeta del bloque, ejecuta `python servidor_aula.py`.
2. Abre `http://127.0.0.1:8765/dinamico`. En DevTools abre Network y recarga.
3. En el documento principal, abre Response y busca AV-101. Anota el estado de la página.
4. Pulsa **Cargar avisos**, filtra Fetch/XHR y abre la respuesta de `/datos.json`.
5. En Elements, localiza el cuerpo de la tabla y comprueba el nuevo estado y las filas.
6. Contrasta /vacio con /mantenimiento. Anota por qué ambos 200 se interpretan de forma distinta.

**Resultado esperado.** Dos registros llegan en JSON después de pulsar. Elements muestra dos filas; el documento inicial sigue sin contener sus IDs. Guarda ruta y observación, sin volcar cabeceras de sesiones externas.

In [7]:
observaciones = [
    ("Response del Doc", "HTML inicial con tabla vacía"),
    ("Fetch/XHR tras pulsar", "JSON con dos IDs"),
    ("Elements tras la carga", "DOM con dos filas"),
]
for vista, hallazgo in observaciones:
    print(vista, "→", hallazgo)

Response del Doc → HTML inicial con tabla vacía
Fetch/XHR tras pulsar → JSON con dos IDs
Elements tras la carga → DOM con dos filas


## Ejemplo 7 · Separar HTTP y resultado de negocio

Un 200 puede acompañar una página de mantenimiento. Un 404 significa que la ruta no se obtuvo como recurso. Conservar el estado y validar una señal de negocio permite separar cero, fallo y pendiente.

**Resultado esperado.** /vacio y /mantenimiento responden 200; sus marcadores difieren. La ruta inexistente responde 404.

In [8]:
with portal() as base:
    for ruta in ["/vacio", "/mantenimiento", "/no-existe"]:
        respuesta = observar(base, ruta)
        marcadores = [m for m in ("vacio", "incidencia")
                      if f'data-estado="{m}"' in texto(respuesta)]
        print(ruta, respuesta["estado"], marcadores)

/vacio 200 ['vacio']
/mantenimiento 200 ['incidencia']
/no-existe 404 []


## Ejemplo 8 · Interpretar robots y condiciones

El RFC 9309 define directivas dirigidas a rastreadores. El recurso /robots.txt del portal indica evitar /interno/. El ejemplo observa únicamente las reglas. El permiso para usar una fuente real se verifica por separado y una ruta pública puede tener condiciones de uso.

**Resultado esperado.** Archivo de texto con un grupo general y una exclusión de la ruta interna.

In [9]:
with portal() as base:
    reglas = observar(base, "/robots.txt")
print(reglas["tipo"])
print(texto(reglas).strip())

text/plain
User-agent: *
Disallow: /interno/


## Ejemplo 9 · Presupuestar peticiones

Por ejecución se pide cada listado y sus detalles. Para páginas homogéneas, multiplica ejecuciones × páginas × (1 + detalles). El modelo es un presupuesto base y excluye recursos auxiliares y reintentos. Define una condición de parada antes de ampliar el recorrido.

**Resultado esperado.** 120 peticiones base si se hacen dos ejecuciones, diez páginas y cinco detalles por página.

In [10]:
ejecuciones, paginas, detalles = 2, 10, 5
presupuesto = ejecuciones * paginas * (1 + detalles)
print(presupuesto)

120


## Ejemplo 10 · Calcular métricas con denominadores distintos

La cobertura usa IDs únicos y un total esperado comparable. La completitud de fecha usa el conjunto observado. Si falta un denominador válido, usa None para indicar desconocido o no calculable y conserva el motivo.

**Resultado esperado.** 70 % de cobertura, 85,7 % de completitud y cobertura desconocida si el total de referencia falta.

In [11]:
observados, esperados, fechas_validas = 7, 10, 6
cobertura = observados / esperados if esperados else None
completitud = fechas_validas / observados if observados else None
print(f"{cobertura:.1%}", f"{completitud:.1%}")
esperados = None
print(observados / esperados if esperados else None)

70.0% 85.7%
None


## Ejemplo 11 · Distinguir alta, cambio y ausencia

Compara capturas mediante identificadores estables. Un ID presente en ambos momentos puede haber cambiado de título. Si el segundo lote es parcial, una ausencia no permite concluir que se retiró el aviso.

**Resultado esperado.** AV-103 es alta, AV-102 cambio y AV-104 ausencia por investigar.

In [12]:
antes = {"AV-101": "A", "AV-102": "B", "AV-104": "E"}
despues = {"AV-101": "A", "AV-102": "C", "AV-103": "D"}
print("Altas:", sorted(despues.keys() - antes.keys()))
print("Cambios:", sorted(k for k in antes.keys() & despues.keys()
                         if antes[k] != despues[k]))
print("Ausencias:", sorted(antes.keys() - despues.keys()))

Altas: ['AV-103']
Cambios: ['AV-102']
Ausencias: ['AV-104']


## Ejemplo 12 · Documentar evidencia y decisión

Guardar URL, estado, tipo, hora y huella ayuda a asociar una decisión a lo observado. El hash verifica igualdad con una evidencia registrada, pero no acredita por sí solo autenticidad ni cobertura. En el ejemplo se guarda solo un resumen en memoria.

**Resultado esperado.** 200, text/html, fecha UTC y una huella de 64 caracteres hexadecimales.

In [13]:
with portal() as base:
    respuesta = observar(base, "/estatico")
evidencia = {k: respuesta[k] for k in ("url", "estado", "tipo", "captura_utc")}
evidencia["sha256"] = sha256(respuesta["bytes"]).hexdigest()
evidencia["decision"] = "Evaluar extracción HTML para estos campos"
print(json.dumps(evidencia, ensure_ascii=False, indent=2))

{
  "url": "http://127.0.0.1:45045/estatico",
  "estado": 200,
  "tipo": "text/html",
  "captura_utc": "2026-11-05T22:26:02.260988+00:00",
  "sha256": "c3b029bfbbb1ca156de4e5e4929008ecc935a7546852365a6374a6acbab88255",
  "decision": "Evaluar extracción HTML para estos campos"
}


## Diez ejercicios autónomos

Completa las celdas y añade una interpretación por ejercicio. Justifica las decisiones con la representación observada, la evidencia disponible y el alcance autorizado.

### Ejercicio 01 · Descomponer una URL

Construye una URL con anio=2026, pagina=2 y fragmento resultados. Recupera los filtros y explica dónde se interpreta el fragmento.

**Pista.** Combina urlencode, urlsplit y parse_qs.

**Criterio de aceptación.** Ruta /avisos, dos filtros y fragmento resultados.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 02 · Comparar las respuestas HTML

Comprueba si AV-102 aparece en /estatico y /dinamico antes de ejecutar scripts. Verifica el tipo de contenido de ambas.

**Pista.** Usa observar y texto dentro de un contexto portal().

**Criterio de aceptación.** Ambas respuestas son HTML. El ID está solo en /estatico.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 03 · Localizar la fuente posterior

Inspecciona /datos.json. Confirma el tipo, los dos IDs y el recuento. Describe en qué momento aparece en Network.

**Pista.** Usa json.loads y la clave items.

**Criterio de aceptación.** Dos registros JSON después de pulsar el botón en el navegador.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 04 · Clasificar las representaciones

Relaciona HTML recibido, Fetch/XHR y DOM actual con una observación del portal. Si tienes navegador, verifica el resultado manualmente.

**Pista.** Usa Response del Doc, Response de datos.json y Elements.

**Criterio de aceptación.** Tres evidencias distintas y una secuencia temporal correcta.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 05 · Diferenciar estado HTTP y disponibilidad

Compara /estatico y /no-existe. Comprueba los códigos y explica qué información adicional necesitas antes de publicar un dato de la primera ruta.

**Pista.** Verifica estado y estructura esperada.

**Criterio de aceptación.** 200 y 404. La publicación exige validar campos y cobertura.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 06 · Distinguir vacío e incidencia

Observa /vacio y /mantenimiento. Usa sus marcadores para clasificar el resultado y explica cuál admite registrar cero para su filtro.

**Pista.** Los dos códigos HTTP son iguales. Busca data-estado.

**Criterio de aceptación.** Solo /vacio declara un vacío explícito.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 07 · Leer el alcance de robots.txt

Recupera /robots.txt, identifica el grupo y la ruta excluida y escribe qué comprobación queda pendiente antes de automatizar una fuente externa.

**Pista.** Localiza User-agent y Disallow. Evita navegar a la ruta excluida.

**Criterio de aceptación.** Grupo general, exclusión /interno/ y autorización pendiente.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 08 · Ajustar un presupuesto

Calcula el coste base de 3 ejecuciones, 8 páginas y 4 detalles por página. Recalcula si el control puede prescindir de los detalles.

**Pista.** Cada página aporta una petición de listado.

**Criterio de aceptación.** 120 frente a 24 peticiones base.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 09 · Medir cobertura con duplicados

Con IDs A, B, B, C, calcula la cobertura sobre cinco avisos esperados y explica por qué contar filas sobreestima la cobertura.

**Pista.** Usa set y compara con el recuento bruto.

**Criterio de aceptación.** Tres IDs únicos y cobertura 60 %. Cuatro filas producirían 80 %.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 10 · Entregar una ficha de diagnóstico

Registra ruta, estado, tipo, instante UTC y SHA-256 para /dinamico antes de pulsar. Formula la conclusión y explica qué técnica evaluarías si el flujo autorizado requiere interacción.

**Pista.** Conserva la huella de los bytes originales y el alcance temporal de la observación.

**Criterio de aceptación.** Ficha completa. El HTML inicial no contiene los avisos y el caso puede pasar a Selenium si la interacción es necesaria.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación: